# Campaign Effectiveness Analysis - Dunnhumby Complete Journey

**Question**: Did households exposed to a major loyalty campaign show a 
statistically significant increase in spend during the campaign period, 
compared to an equivalent period before it and was this specific to the 
campaign, or part of a broader spending trend?

In [4]:
!pip install duckdb


In [5]:
import pandas as pd
import duckdb
import os
print(os.listdir())

['hh_demographic.csv', 'causal_data.csv', 'campaign_analysis.ipynb', 'coupon.csv', 'campaign_table.csv', 'campaign_effectiveness_analysis.ipynb', '.ipynb_checkpoints', 'coupon_redempt.csv', 'product.csv', 'campaign_desc.csv', 'transaction_data.csv']


In [6]:
campaign_desc = pd.read_csv('campaign_desc.csv')
campaign_table = pd.read_csv('campaign_table.csv')

print(campaign_desc.head())
print(campaign_desc.shape)

  DESCRIPTION  CAMPAIGN  START_DAY  END_DAY
0       TypeB        24        659      719
1       TypeC        15        547      708
2       TypeB        25        659      691
3       TypeC        20        615      685
4       TypeB        23        646      684
(30, 4)


## Selecting a campaign
Campaigns vary widely in size. I selected the largest campaign by household 
count to maximise statistical power (Campaign 18).

In [7]:
query1 = """
SELECT 
    CAMPAIGN,
    COUNT(DISTINCT household_key) AS num_households
FROM 'campaign_table.csv'
GROUP BY CAMPAIGN
ORDER BY num_households DESC
"""
result1 = duckdb.sql(query1).df()
print(result1.head(10))

   CAMPAIGN  num_households
0        18            1133
1        13            1077
2         8            1076
3        30             361
4        26             332
5        22             276
6        20             244
7        14             224
8        11             214
9        17             202


## Defining before/during windows
Campaign 18 ran for 55 days (day 587–642). I used an equal-length baseline 
window immediately prior (day 532–587) for a fair before/after comparison.

In [8]:
query2 = """
SELECT START_DAY, END_DAY
FROM 'campaign_desc.csv'
WHERE CAMPAIGN = 18
"""
result2 = duckdb.sql(query2).df()
print(result2)


   START_DAY  END_DAY
0        587      642


In [9]:
query3 = """
SELECT household_key
FROM 'campaign_table.csv'
WHERE CAMPAIGN = 18
"""
result3 = duckdb.sql(query3).df()
print(result3.shape)
print(result3.head())

(1133, 1)
   household_key
0              1
1              6
2              7
3              8
4             14


In [10]:
transactions = duckdb.sql("SELECT * FROM 'transaction_data.csv' LIMIT 5").df()
print(transactions)

   household_key    BASKET_ID  DAY  PRODUCT_ID  QUANTITY  SALES_VALUE  \
0           2375  26984851472    1     1004906         1         1.39   
1           2375  26984851472    1     1033142         1         0.82   
2           2375  26984851472    1     1036325         1         0.99   
3           2375  26984851472    1     1082185         1         1.21   
4           2375  26984851472    1     8160430         1         1.50   

   STORE_ID  RETAIL_DISC TRANS_TIME  WEEK_NO  COUPON_DISC  COUPON_MATCH_DISC  
0       364        -0.60       1631        1          0.0                0.0  
1       364         0.00       1631        1          0.0                0.0  
2       364        -0.30       1631        1          0.0                0.0  
3       364         0.00       1631        1          0.0                0.0  
4       364        -0.39       1631        1          0.0                0.0  


In [11]:
query = """
SELECT
    household_key,
    CASE 
        WHEN DAY >= 532 AND DAY < 587 THEN 'before'
        WHEN DAY >= 587 AND DAY < 642 THEN 'during'
    END AS period,
    SUM(SALES_VALUE) AS total_spend
FROM 'transaction_data.csv'
WHERE household_key IN (
    SELECT household_key FROM 'campaign_table.csv' WHERE CAMPAIGN = 18
)
AND DAY >= 532 AND DAY < 642
GROUP BY household_key, period
"""

result = duckdb.sql(query).df()
print(result.head(10))
print(result.shape)

   household_key  period  total_spend
0           1111  before      1584.64
1           1591  before       711.35
2           1222  before       440.07
3            916  before       572.06
4           1740  before      1452.40
5           1453  before      1645.21
6           1804  before       718.39
7           2129  before       615.17
8           1038  before       176.96
9           2071  before       157.33
(2219, 3)


## Reshaping and handling missing periods
Households with no transactions in a given period are treated as £0 spend 
(not dropped), since they may represent customers activated by the campaign.

In [12]:
pivot = result.pivot(index='household_key', columns='period', values='total_spend')
pivot = pivot.fillna(0)  # households with no purchases in a period = £0 spend, not missing
print(pivot.head(10))
print(pivot.shape)

period          before   during
household_key                  
1               342.71   465.67
2               236.35   201.86
6               465.11   406.07
7               392.31   299.41
8               464.81   478.20
13             1092.75  1538.71
14              148.96   132.54
17              154.72   485.42
18              583.89   872.78
19             1424.94  1328.67
(1123, 2)


In [13]:
pivot['change'] = pivot['during'] - pivot['before']
print(f"Average spend before: £{pivot['before'].mean():.2f}")
print(f"Average spend during: £{pivot['during'].mean():.2f}")
print(f"Average change: £{pivot['change'].mean():.2f}")

Average spend before: £430.86
Average spend during: £465.63
Average change: £34.77


## Testing significance
Used a paired t-test since we're comparing each household against itself 
across two time periods.

In [14]:
from scipy import stats

t_stat, p_value = stats.ttest_rel(pivot['during'], pivot['before'])
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.5f}")

t-statistic: 5.423
p-value: 0.00000


## Why a control group is needed

A before/after comparison alone cannot tell us whether the increase in spend 
was caused by the campaign, or reflects a broader trend affecting all 
customers during this period (e.g. seasonality). To test this, I compared 
the same before/during change for a control group of households not exposed 
to any campaign running during this window.

## Identifying overlapping campaigns

Several other campaigns overlapped the analysis window. To build a clean 
control group, I excluded households in any of these campaigns, not just 
Campaign 18.

In [15]:
overlapping = duckdb.sql("""
    SELECT CAMPAIGN, START_DAY, END_DAY
    FROM 'campaign_desc.csv'
    WHERE START_DAY < 642 AND END_DAY > 532
""").df()
print(overlapping)

   CAMPAIGN  START_DAY  END_DAY
0        15        547      708
1        20        615      685
2        21        624      656
3        22        624      656
4        18        587      642
5        19        603      635
6        17        575      607
7        14        531      596
8        16        561      593
9        13        504      551


In [16]:
query_control = """
SELECT
    household_key,
    CASE 
        WHEN DAY >= 532 AND DAY < 587 THEN 'before'
        WHEN DAY >= 587 AND DAY < 642 THEN 'during'
    END AS period,
    SUM(SALES_VALUE) AS total_spend
FROM 'transaction_data.csv'
WHERE household_key NOT IN (
    SELECT household_key FROM 'campaign_table.csv' 
    WHERE CAMPAIGN IN (13,14,15,16,17,18,19,20,21,22)
)
AND DAY >= 532 AND DAY < 642
GROUP BY household_key, period
"""

control_result = duckdb.sql(query_control).df()
print(control_result.shape)

(1744, 3)


## Calculating control group spend


In [17]:
control_pivot = control_result.pivot(index='household_key', columns='period', values='total_spend')
control_pivot = control_pivot.fillna(0)
control_pivot['change'] = control_pivot['during'] - control_pivot['before']

print(f"Control households: {control_pivot.shape[0]}")
print(f"Average spend before: £{control_pivot['before'].mean():.2f}")
print(f"Average spend during: £{control_pivot['during'].mean():.2f}")
print(f"Average change: £{control_pivot['change'].mean():.2f}")

t_stat_control, p_value_control = stats.ttest_rel(control_pivot['during'], control_pivot['before'])
print(f"t-statistic: {t_stat_control:.3f}")
print(f"p-value: {p_value_control:.5f}")

Control households: 978
Average spend before: £119.26
Average spend during: £140.92
Average change: £21.66
t-statistic: 5.459
p-value: 0.00000


**Findings**:
- Campaign 18 households: average spend rose from £430.86 to £465.63 
  (+£34.77, +8.1%), statistically significant (p < 0.001)
- Control households: average spend rose from £119.26 to £140.92 
  (+£21.66, +18.2%), also statistically significant (p < 0.001)

**Interpretation**: Both groups increased spend over this period, suggesting 
some of the rise reflects a general trend (e.g. seasonality) rather than the 
campaign alone. Campaign households increased more in absolute pounds, but 
less in percentage terms. Campaign households also had 3.6x higher baseline 
spend than the control group before the campaign even started, indicating 
Campaign 18 was not targeted at a random sample of customers, but likely at 
already high-value shoppers — a clear instance of selection bias.

**Limitation**: This analysis cannot cleanly separate the campaign's effect 
from broader spending trends or pre-existing differences between the two 
groups. A stronger analysis would match campaign and control households on 
similar baseline spend before comparing their changes, to better isolate the 
campaign's true incremental effect.

**Conclusion**: The data does not provide clear evidence that Campaign 18 
caused incremental spend beyond what would have happened anyway. The result 
is genuinely ambiguous — campaign households show a larger absolute increase, 
but a smaller relative increase, and started from a very different baseline. 
Further work using baseline-matched control groups would be needed to draw a 
confident conclusion about the campaign's true effect.

## Addressing selection bias: baseline-matched comparison

The unmatched control group showed campaign households had a 3.6x higher 
baseline spend than control households — meaning campaign targeting was not 
random, and the two groups were not directly comparable. To address this, I 
restricted both groups to a shared baseline spend range and re-ran the 
comparison on this matched subset.

In [18]:
print(pivot['before'].describe())
print(control_pivot['before'].describe())

count    1123.000000
mean      430.861006
std       350.391222
min         0.000000
25%       177.950000
50%       343.820000
75%       573.595000
max      2652.550000
Name: before, dtype: float64
count     978.000000
mean      119.259039
std       200.572430
min         0.000000
25%        16.965000
50%        59.420000
75%       152.105000
max      4093.570000
Name: before, dtype: float64


Checking the overlap between groups confirmed limited but real comparability 
in the £100–£300 baseline spend range.

In [19]:
print(f"Campaign households with 'before' spend under £200: {(pivot['before'] < 200).sum()} out of {len(pivot)}")
print(f"Control households with 'before' spend over £200: {(control_pivot['before'] > 200).sum()} out of {len(control_pivot)}")

Campaign households with 'before' spend under £200: 316 out of 1123
Control households with 'before' spend over £200: 176 out of 978


In [20]:
matched_campaign = pivot[(pivot['before'] >= 100) & (pivot['before'] <= 300)]
matched_control = control_pivot[(control_pivot['before'] >= 100) & (control_pivot['before'] <= 300)]

print(matched_campaign.shape)
print(matched_control.shape)

(320, 3)
(266, 3)


In [21]:
t_stat_matched, p_value_matched = stats.ttest_rel(matched_campaign['during'], matched_campaign['before'])

print("MATCHED CAMPAIGN GROUP (before spend £100-£300)")
print(f"  Before: £{matched_campaign['before'].mean():.2f} → During: £{matched_campaign['during'].mean():.2f}")
print(f"  Change: £{(matched_campaign['during'] - matched_campaign['before']).mean():.2f}")
print(f"  p-value: {p_value_matched:.5f}")

t_stat_matched_control, p_value_matched_control = stats.ttest_rel(matched_control['during'], matched_control['before'])

print("\nMATCHED CONTROL GROUP (before spend £100-£300)")
print(f"  Before: £{matched_control['before'].mean():.2f} → During: £{matched_control['during'].mean():.2f}")
print(f"  Change: £{(matched_control['during'] - matched_control['before']).mean():.2f}")
print(f"  p-value: {p_value_matched_control:.5f}")

MATCHED CAMPAIGN GROUP (before spend £100-£300)
  Before: £196.89 → During: £248.68
  Change: £51.79
  p-value: 0.00000

MATCHED CONTROL GROUP (before spend £100-£300)
  Before: £170.91 → During: £176.37
  Change: £5.46
  p-value: 0.52078


## Updated Conclusion

**Question**: Did households exposed to Campaign 18 show a real increase in 
spend during the campaign period, or would they have spent that money anyway?

**Method**: Compared before/during spend for campaign households, then for a 
control group unexposed to any overlapping campaign. Found campaign 
households had 3.6x higher baseline spend than control households, 
indicating non-random targeting. Restricted both groups to a shared baseline 
spend band (£100–£300) and re-ran the comparison on this matched subset.

**Findings**:
- Unmatched: both campaign (+8.1%) and control (+18.2%) groups rose 
  significantly — ambiguous, likely confounded by baseline differences.
- Matched (£100–£300 baseline): campaign group rose significantly 
  (+£51.79, p<0.001); control group showed no significant change 
  (+£5.46, p=0.52).

**Interpretation**: Once baseline spend differences were controlled for, the 
control group's apparent increase disappeared, while the campaign group's 
increase held. This is strong evidence that Campaign 18 drove real 
incremental spend, at least for mid-spend households.

**Limitation**: This matched result applies only to the £100–£300 baseline 
band (28% of the original campaign group) — there wasn't enough control 
group overlap to test very low or very high spenders. A regression-based 
approach could extend this analysis across the full spend range.

**Conclusion**: Campaign 18 shows genuine evidence of driving incremental 
spend among mid-spend households, once selection bias from non-random 
targeting is controlled for. This finding is more reliable than the initial 
unmatched comparison, which was confounded by large baseline differences 
between groups.